# Day 3: Time Series Analysis with Pandas

## Learning Objectives
By the end of this notebook, you will be able to:
- Convert strings to datetime objects
- Extract date components (year, month, quarter, day of week)
- Filter data by date ranges
- Sort data chronologically
- Calculate rolling averages and moving statistics
- Perform time-based aggregations
- Analyze trends over time

**Duration**: 45 minutes

---

## 1. Why Time Series Analysis?

Most business data has a time component:
- Daily sales
- Monthly revenue
- Quarterly targets
- Year-over-year growth

Pandas makes working with dates and time incredibly powerful!

### Demo 1.1: The Problem with Date Strings

In [ ]:
# Demo: The Problem with Date Strings
# Dates stored as strings can't do date math!
# You need to convert them to datetime objects

import pandas as pd

# Dates stored as strings (common when reading CSV files)
df = pd.DataFrame({
    'Date': ['2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18'],
    'Revenue': [1500, 2000, 1800, 2200]
})

# Check data types - Date shows as 'object' (string)
print("Data types:")
print(df.dtypes)
print("\nData:")
print(df)

# Problem: Can't extract month, day, etc. from strings!
# df['Month'] = df['Date'].month  # This would ERROR!

print("\nProblem: Date is 'object' (string), not datetime!")

### Demo 1.2: Converting to Datetime

In [ ]:
# Demo: Converting Strings to Datetime
# pd.to_datetime() converts strings to datetime objects
# Syntax: df["column"] = pd.to_datetime(df["column"])

# Convert the Date column
df['Date'] = pd.to_datetime(df['Date'])

print("After conversion:")
print(df.dtypes)  # Now shows datetime64[ns]
print("\nNow Date is datetime64! We can do date operations.")
print(df)

---
## 2. Extracting Date Components

Use the `.dt` accessor to extract parts of dates.

### Demo 2.1: Basic Date Components

In [ ]:
# Demo: Extracting Date Components
# Use the .dt accessor to extract parts of dates
# .dt gives access to datetime properties and methods
# Syntax: df["date_column"].dt.property

# Create sample data with dates
df = pd.DataFrame({
    'Date': pd.to_datetime(['2024-01-15', '2024-02-20', '2024-03-10', 
                            '2024-04-05', '2024-05-18', '2024-06-22']),
    'Revenue': [15000, 18000, 16500, 19000, 17500, 21000]
})

# Extract various date components using .dt accessor
df['Year'] = df['Date'].dt.year          # Extract year (2024)
df['Month'] = df['Date'].dt.month        # Extract month number (1-12)
df['Day'] = df['Date'].dt.day            # Extract day of month (1-31)
df['DayOfWeek'] = df['Date'].dt.day_name()  # Get day name (Monday, Tuesday, etc.)

print("With date components:")
print(df)

### Demo 2.2: Business-Relevant Components

In [ ]:
# Demo: Business-Relevant Date Components
# Extract components useful for business analysis

# Quarter (1-4) - essential for quarterly reporting
df['Quarter'] = df['Date'].dt.quarter

# Month name - more readable than numbers
df['MonthName'] = df['Date'].dt.month_name()

# Week of year (1-52) - useful for weekly analysis
df['WeekOfYear'] = df['Date'].dt.isocalendar().week

print("With business components:")
print(df[['Date', 'Revenue', 'Quarter', 'MonthName', 'DayOfWeek']])

### Exercise 1: Extract Date Components

Load sales data and:
1. Convert Date column to datetime
2. Extract Year, Month, and Quarter
3. Group by Quarter and calculate total revenue
4. Find which quarter had highest revenue

In [ ]:
# Exercise 1: Extract Date Components
# Task: Convert dates and extract year, month, quarter

# Sample data - dates as strings
sales = pd.DataFrame({
    'Date': ['2024-01-15', '2024-02-20', '2024-03-10', '2024-04-05', 
             '2024-05-18', '2024-06-22', '2024-07-14', '2024-08-09',
             '2024-09-25', '2024-10-12', '2024-11-08', '2024-12-19'],
    'Revenue': [15000, 18000, 16500, 19000, 17500, 21000, 
                19500, 22000, 20500, 23000, 21500, 24000]
})

# 1. Convert to datetime
# Use pd.to_datetime() on the Date column
sales['Date'] = pd.to_datetime(sales['___'])  # 'Date'

# 2. Extract components using .dt accessor
sales['Year'] = sales['Date'].dt.___      # year
sales['Month'] = sales['Date'].dt.___     # month
sales['Quarter'] = sales['Date'].dt.___   # quarter

print("Sales with date components:")
print(sales)

# 3. Group by quarter and sum revenue
quarterly_revenue = sales.groupby('Quarter')['Revenue'].sum()
print("\nQuarterly revenue:")
print(quarterly_revenue)

# 4. Find best quarter using idxmax()
best_quarter = quarterly_revenue.idxmax()
best_revenue = quarterly_revenue.max()
print(f"\nBest quarter: Q{best_quarter} with €{best_revenue:,}")

---
## 3. Filtering by Date Ranges

Select data within specific time periods.

### Demo 3.1: Simple Date Filtering

In [ ]:
# Demo: Filtering by Date Ranges
# Compare dates using standard comparison operators

# Create larger dataset - pd.date_range() creates sequence of dates
dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')  # Daily frequency
df = pd.DataFrame({
    'Date': dates,
    'Revenue': range(1000, 1000 + len(dates))
})

print(f"Total records: {len(df)}")

# Filter for Q1 (January to March)
# Use string dates - pandas automatically converts them for comparison
q1 = df[(df['Date'] >= '2024-01-01') & (df['Date'] <= '2024-03-31')]
print(f"\nQ1 records: {len(q1)}")
print(q1.head())

# Filter for specific month using .dt.month
june = df[df['Date'].dt.month == 6]  # Month 6 = June
print(f"\nJune records: {len(june)}")

### Demo 3.2: Between Method for Dates

In [ ]:
# Demo: Using .between() for Date Ranges
# .between() is cleaner than combining >= and <= conditions
# Syntax: df[df["date_column"].between("start", "end")]

# Filter Q2 using between() - more readable!
q2 = df[df['Date'].between('2024-04-01', '2024-06-30')]

print(f"Q2 data: {len(q2)} days")
print(f"Q2 total revenue: €{q2['Revenue'].sum():,}")
print(f"Q2 average daily: €{q2['Revenue'].mean():,.2f}")

### Exercise 2: Date Range Analysis

Given daily sales data:
1. Filter for all sales in May 2024
2. Filter for first half of 2024 (Jan-Jun)
3. Compare H1 vs H2 total revenue
4. Calculate growth rate

In [ ]:
# Exercise 2: Date Range Analysis
# Task: Compare first half vs second half of 2024

# Daily data for all of 2024
dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')
daily_sales = pd.DataFrame({
    'Date': dates,
    'Revenue': [1500 + i*5 for i in range(len(dates))]  # Increasing trend
})

# 1. May sales - filter by month number
may_sales = daily_sales[daily_sales['Date'].dt.month == ___]  # 5
print(f"May sales: {len(may_sales)} days, €{may_sales['Revenue'].sum():,}")

# 2. H1 (first half) - use .between()
h1 = daily_sales[daily_sales['Date'].between(___, ___)]  # '2024-01-01', '2024-06-30'
h1_total = h1['Revenue'].sum()

# H2 (second half)
h2 = daily_sales[daily_sales['Date'].between('2024-07-01', '2024-12-31')]
h2_total = h2['Revenue'].sum()

# 3-4. Compare H1 vs H2
growth = ((h2_total - h1_total) / h1_total) * 100

print(f"\nH1 Total: €{h1_total:,}")
print(f"H2 Total: €{h2_total:,}")
print(f"H2 vs H1 Growth: {growth:+.1f}%")

---
## 4. Sorting by Date

Always sort time series data chronologically!

### Demo 4.1: Sorting Time Series

In [ ]:
# Demo: Sorting Time Series Data
# ALWAYS sort by date for time series analysis!
# Syntax: df.sort_values("date_column")

# Unsorted data (common when data comes from different sources)
df = pd.DataFrame({
    'Date': pd.to_datetime(['2024-03-15', '2024-01-10', '2024-02-20', '2024-01-05']),
    'Revenue': [1600, 1500, 1800, 1400]
})

print("Unsorted:")
print(df)

# Sort chronologically by date
df_sorted = df.sort_values('Date')
print("\nSorted chronologically:")
print(df_sorted)

# Reset index after sorting (optional but cleaner)
# drop=True prevents old index from becoming a column
df_sorted = df_sorted.reset_index(drop=True)
print("\nWith reset index:")
print(df_sorted)

---
## 5. Rolling Calculations (Moving Averages)

Smooth out fluctuations to see trends!

### Demo 5.1: Rolling Average

In [ ]:
# Demo: Rolling Average (Moving Average)
# Rolling calculations smooth out daily fluctuations to reveal trends
# Syntax: df["column"].rolling(window=n).function()
# window = number of periods to include in each calculation

import numpy as np

# Create daily sales with random fluctuations
dates = pd.date_range('2024-01-01', periods=30, freq='D')
df = pd.DataFrame({
    'Date': dates,
    'DailySales': [2000 + np.random.randint(-300, 500) for _ in range(30)]
})

# Calculate 7-day rolling average
# Each value is the average of the current row and previous 6 rows
df['Rolling_7Day_Avg'] = df['DailySales'].rolling(window=7).mean()

print("With 7-day rolling average:")
print(df.head(10))
print("\nNote: First 6 rows are NaN (not enough data for 7-day window)")

### Demo 5.2: Rolling Sum and Other Stats

In [ ]:
# Demo: Multiple Rolling Statistics
# You can calculate sum, max, min, etc. over rolling windows

# Add various rolling calculations
df['Rolling_7Day_Sum'] = df['DailySales'].rolling(window=7).sum()   # 7-day total
df['Rolling_7Day_Max'] = df['DailySales'].rolling(window=7).max()   # 7-day peak
df['Rolling_7Day_Min'] = df['DailySales'].rolling(window=7).min()   # 7-day minimum

print("Multiple rolling statistics:")
print(df[['Date', 'DailySales', 'Rolling_7Day_Avg', 'Rolling_7Day_Max']].tail(10))

### Exercise 3: Rolling Analysis

Given monthly revenue data:
1. Calculate 3-month rolling average
2. Find the month where 3-month average was highest
3. Calculate rolling sum (3-month totals)

In [ ]:
# Exercise 3: Rolling Analysis
# Task: Calculate rolling statistics on monthly data

# Monthly revenue data (2 years)
months = pd.date_range('2023-01-01', periods=24, freq='MS')  # MS = Month Start
monthly_revenue = pd.DataFrame({
    'Month': months,
    'Revenue': [15000, 16000, 14500, 17000, 18500, 19000, 20500, 21000,
                19500, 22000, 23500, 25000, 24500, 26000, 25500, 27000,
                28500, 29000, 30500, 31000, 30000, 32000, 33500, 35000]
})

# 1. Calculate 3-month rolling average
# Use .rolling(window=n).mean()
monthly_revenue['Rolling_3Month_Avg'] = monthly_revenue['Revenue'].rolling(window=___).mean()  # 3

# 2. Find month with highest 3-month average
best_idx = monthly_revenue['Rolling_3Month_Avg'].idxmax()  # Index of max value
best_month = monthly_revenue.loc[best_idx, 'Month']
best_avg = monthly_revenue.loc[best_idx, 'Rolling_3Month_Avg']

print(f"Highest 3-month average: €{best_avg:,.2f}")
print(f"Month: {best_month.strftime('%B %Y')}")

# 3. Rolling sum (3-month totals)
# Use .rolling(window=3).sum()
monthly_revenue['Rolling_3Month_Sum'] = monthly_revenue['Revenue'].rolling(window=3).___  # sum()

print("\nRecent data:")
print(monthly_revenue[['Month', 'Revenue', 'Rolling_3Month_Avg', 'Rolling_3Month_Sum']].tail())

---
## 6. Time-Based Aggregations

Summarize data by time periods.

### Demo 6.1: Monthly Aggregation from Daily Data

In [ ]:
# Demo: Time-Based Aggregation from Daily to Monthly
# Convert daily data to monthly summaries

# Daily sales data for a year
dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')
daily = pd.DataFrame({
    'Date': dates,
    'Revenue': [1500 + i*2 for i in range(len(dates))]
})

# Method 1: Extract month and groupby
# .dt.to_period('M') converts to month periods (e.g., '2024-01')
daily['Month'] = daily['Date'].dt.to_period('M')
monthly_summary = daily.groupby('Month')['Revenue'].agg(['sum', 'mean', 'count'])

print("Monthly summary from daily data:")
print(monthly_summary.head())

### Demo 6.2: Resample Method

In [ ]:
# Demo: Using resample() for Time Aggregation
# resample() is the preferred method for time-based aggregation
# Requires date column as index
# Syntax: df.resample('frequency').aggregation()

# Set Date as index (required for resample)
daily_indexed = daily.set_index('Date')

# Resample to monthly totals
# 'M' = month end frequency
monthly = daily_indexed['Revenue'].resample('M').sum()

print("Using resample (cleaner):")
print(monthly.head())

# Resample to quarterly totals
# 'Q' = quarter end frequency
quarterly = daily_indexed['Revenue'].resample('Q').sum()
print("\nQuarterly:")
print(quarterly)

### Exercise 4: Time-Based Aggregation

Given daily transaction data:
1. Calculate weekly totals
2. Calculate monthly averages
3. Find best week and best month
4. Calculate quarter-over-quarter growth

In [ ]:
# Exercise 4: Time-Based Aggregation
# Task: Aggregate daily transactions to weekly and monthly

import numpy as np

# Daily transaction data for 2024
dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')
transactions = pd.DataFrame({
    'Date': dates,
    'Amount': [2000 + i*3 + np.random.randint(-200, 300) for i in range(len(dates))]
})

# Set Date as index for resample
trans_indexed = transactions.set_index('Date')

# 1. Weekly totals using resample
# 'W' = weekly frequency
weekly = trans_indexed['Amount'].resample('W').___  # sum()
print("Weekly totals (first 5 weeks):")
print(weekly.head())

# 2. Monthly averages
# Use .mean() for average
monthly = trans_indexed['Amount'].resample('M').___  # mean()
print("\nMonthly averages:")
print(monthly)

# 3. Find best week and best month using idxmax()
best_week = weekly.idxmax()
best_month = monthly.idxmax()
print(f"\nBest week: {best_week.strftime('%Y-%m-%d')} (€{weekly.max():,.2f})")
print(f"Best month: {best_month.strftime('%B %Y')} (€{monthly.max():,.2f})")

# 4. Quarter-over-quarter growth
# pct_change() calculates percentage change from previous period
quarterly = trans_indexed['Amount'].resample('Q').sum()
qoq_growth = quarterly.pct_change() * 100  # Convert to percentage
print("\nQuarter-over-quarter growth:")
print(qoq_growth)

---
## Challenge Exercise: Complete Time Series Analysis

Given 2 years of daily sales data:
1. Load and prepare the data (convert dates, sort)
2. Calculate month-over-month growth
3. Add 30-day rolling average
4. Compare Q4 2024 vs Q4 2023
5. Find the best 90-day period (highest rolling sum)

In [ ]:
# Create 2-year dataset
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
sales_data = pd.DataFrame({
    'Date': dates,
    'DailySales': [1800 + i*2 + np.random.randint(-300, 400) for i in range(len(dates))]
})

# Your solution here
# 1. Prepare data
sales_data['Date'] = pd.to_datetime(sales_data['Date'])
sales_data = sales_data.sort_values('Date').reset_index(drop=True)

# 2. Month-over-month growth
sales_data['Month'] = sales_data['Date'].dt.to_period('M')
monthly = sales_data.groupby('Month')['DailySales'].sum()
mom_growth = monthly.pct_change() * 100

print("Month-over-month growth (last 6 months):")
print(mom_growth.tail(6))

# 3. 30-day rolling average
sales_data['Rolling_30Day'] = sales_data['DailySales'].rolling(window=30).mean()

# 4. Q4 comparison
q4_2023 = sales_data[(sales_data['Date'] >= '2023-10-01') & 
                      (sales_data['Date'] <= '2023-12-31')]['DailySales'].sum()
q4_2024 = sales_data[(sales_data['Date'] >= '2024-10-01') & 
                      (sales_data['Date'] <= '2024-12-31')]['DailySales'].sum()
q4_growth = ((q4_2024 - q4_2023) / q4_2023) * 100

print(f"\nQ4 2023: €{q4_2023:,.2f}")
print(f"Q4 2024: €{q4_2024:,.2f}")
print(f"YoY Growth: {q4_growth:+.1f}%")

# 5. Best 90-day period
sales_data['Rolling_90Day_Sum'] = sales_data['DailySales'].rolling(window=90).sum()
best_idx = sales_data['Rolling_90Day_Sum'].idxmax()
best_date = sales_data.loc[best_idx, 'Date']
best_total = sales_data.loc[best_idx, 'Rolling_90Day_Sum']

print(f"\nBest 90-day period ending: {best_date.strftime('%Y-%m-%d')}")
print(f"Total: €{best_total:,.2f}")

---
## Summary

**You've learned:**
- Converting strings to datetime with `pd.to_datetime()`
- Extracting date components using `.dt` accessor
- Filtering by date ranges
- Sorting time series chronologically
- Rolling calculations for trend analysis
- Time-based aggregations with resample()
- Growth calculations over time

**Key Takeaways:**
- Always convert date strings to datetime type
- Use `.dt` accessor for date component extraction
- Sort data chronologically before analysis
- Rolling windows smooth out fluctuations
- resample() is powerful for period aggregations
- Time series analysis reveals trends and patterns

**Next:** Working with Excel files for reporting!